In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

file_path = project_path / "data" / "processed" / "symptom2disease_clean.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())

Dataset shape: (1153, 2)

Columns: ['label', 'text']


,label,text
0,Psoriasis,I have been experiencing a skin rash on my arm...
1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,Psoriasis,I have been experiencing joint pain in my fing...
3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,Psoriasis,"My nails have small dents or pits in them, and..."


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["text"])

similarity_matrix = cosine_similarity(X)

print("TF-IDF shape:", X.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

TF-IDF shape: (1153, 8844)
Similarity matrix shape: (1153, 1153)


In [3]:
threshold = 0.85

i, j = np.triu_indices(len(df), k=1)

scores = similarity_matrix[i, j]
selected = scores >= threshold

similar_pairs = pd.DataFrame({
    "index_1": i[selected],
    "index_2": j[selected],
    "similarity": scores[selected]
})

similar_pairs = similar_pairs.sort_values(
    "similarity",
    ascending=False
)

print("Pairs with similarity >= 0.85:", len(similar_pairs))

for _, row in similar_pairs.head(5).iterrows():
    a = df.iloc[int(row["index_1"])]
    b = df.iloc[int(row["index_2"])]

    print("\nSimilarity:", round(row["similarity"], 3))
    print("Labels:", a["label"], "|", b["label"])
    print("Text 1:", a["text"])
    print("Text 2:", b["text"])

Pairs with similarity >= 0.85: 32

Similarity: 1.0
Labels: Acne | Acne
Text 1: I've recently developed a terrible rash on my skin. It's riddled with blackheads and pus-filled pimples. My skin has also been scurrying everywhere.
Text 2: I've recently developed a terrible rash on my skin. It's riddled with blackheads and pus-filled pimples. My skin has also been scurrying about.

Similarity: 1.0
Labels: Acne | Acne
Text 1: I woke up this morning and found that I have a really nasty rash all over my skin. There are a ton of blackheads and pimples that are filled with pus. My skin has also been really scurrying.
Text 2: I woke up this morning to find a really nasty rash on my skin. There are a ton of blackheads and pimples that are filled with pus. My skin has also been really scurrying.

Similarity: 1.0
Labels: Migraine | Migraine
Text 1: I have been experiencing acidity, indigestion, headaches, and blurred and distorted vision, as well as excessive hunger, a stiff neck, depression, irrit

In [4]:
from collections import Counter

# Krijojme nje grup fillestar per cdo tekst
parent = list(range(len(df)))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

# Bashkojme tekstet me similarity >= 0.85
for _, row in similar_pairs.iterrows():
    a = int(row["index_1"])
    b = int(row["index_2"])
    union(a, b)

# Caktojme nje group_id per secilin tekst
df["group_id"] = [find(i) for i in range(len(df))]

group_sizes = df["group_id"].value_counts()

print("Total descriptions:", len(df))
print("Total groups:", df["group_id"].nunique())
print("Groups with multiple texts:", (group_sizes > 1).sum())
print("Largest group size:", group_sizes.max())

print("\nGroup size distribution:")
print(group_sizes.value_counts().sort_index())

Total descriptions: 1153
Total groups: 1124
Groups with multiple texts: 23
Largest group size: 4

Group size distribution:
count
1    1101
2      18
3       4
4       1
Name: count, dtype: int64


In [5]:
group_labels = df.groupby("group_id")["label"].nunique()

mixed_groups = group_labels[group_labels > 1]

print("Groups with different disease labels:",
      len(mixed_groups))

if len(mixed_groups) > 0:
    display(
        df[df["group_id"].isin(mixed_groups.index)]
        [["group_id", "label", "text"]]
        .sort_values("group_id")
    )

Groups with different disease labels: 0


In [6]:
from sklearn.model_selection import GroupShuffleSplit

all_labels = set(df["label"].unique())

def create_splits(seed):
    # Ndarja e pare: 85% train+validation, 15% test
    splitter1 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=seed
    )

    train_val_idx, test_idx = next(
        splitter1.split(df, groups=df["group_id"])
    )

    train_val = df.iloc[train_val_idx].copy()
    test = df.iloc[test_idx].copy()

    # Ndarja e dyte: 70% train, 15% validation
    splitter2 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15 / 0.85,
        random_state=seed
    )

    train_idx, val_idx = next(
        splitter2.split(
            train_val,
            groups=train_val["group_id"]
        )
    )

    train = train_val.iloc[train_idx].copy()
    val = train_val.iloc[val_idx].copy()

    return train, val, test


# Kerkome nje ndarje qe ruan te 24 klasat
for seed in range(200):
    train_df, val_df, test_df = create_splits(seed)

    if all(
        set(part["label"].unique()) == all_labels
        for part in [train_df, val_df, test_df]
    ):
        break
else:
    raise ValueError(
        "Nuk u gjet ndarje me te gjitha klasat."
    )

print("Random seed:", seed)
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Random seed: 0
Training: (803, 3)
Validation: (170, 3)
Testing: (180, 3)


In [7]:
train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])
test_groups = set(test_df["group_id"])

print("Train-Val overlap:",
      len(train_groups & val_groups))

print("Train-Test overlap:",
      len(train_groups & test_groups))

print("Val-Test overlap:",
      len(val_groups & test_groups))

print("\nTotal rows:",
      len(train_df) + len(val_df) + len(test_df))

for name, part in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(
        name,
        "- rows:", len(part),
        "- classes:", part["label"].nunique()
    )

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

Total rows: 1153
Training - rows: 803 - classes: 24
Validation - rows: 170 - classes: 24
Testing - rows: 180 - classes: 24


In [8]:
distribution = pd.DataFrame({
    "Training": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Testing": test_df["label"].value_counts()
}).fillna(0).astype(int)

display(distribution)

print("\nTotal:")
print(distribution.sum())

,Training,Validation,Testing
label,,,
Acne,41,1,4
Arthritis,29,8,9
Bronchial Asthma,30,11,8
Cervical spondylosis,34,6,9
Chicken pox,36,5,8
Common Cold,29,11,9
Dengue,34,10,6
Dimorphic Hemorrhoids,27,7,7
Fungal infection,38,8,4



Total:
Training      803
Validation    170
Testing       180
dtype: int64


In [9]:
processed_path = (
    project_path / "data" / "processed"
)

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

train_df.to_csv(
    processed_path / "train.csv",
    index=False
)

val_df.to_csv(
    processed_path / "validation.csv",
    index=False
)

test_df.to_csv(
    processed_path / "test.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!


In [10]:
import numpy as np
import pandas as pd

best_score = float("inf")
best_result = None
best_seed = None

target = np.array([0.70, 0.15, 0.15])
labels = sorted(df["label"].unique())

for current_seed in range(500):
    train, val, test = create_splits(current_seed)

    parts = [train, val, test]

    counts = np.array([
        [part["label"].value_counts().get(label, 0)
         for part in parts]
        for label in labels
    ])

    # Minimum 3 shembuj per klase ne secilen pjese
    if counts.min() < 3:
        continue

    proportions = counts / counts.sum(axis=1, keepdims=True)

    # Mat devijimin nga ndarja 70/15/15
    score = np.abs(proportions - target).mean()

    if score < best_score:
        best_score = score
        best_result = (train, val, test)
        best_seed = current_seed

if best_result is None:
    print("Nuk u gjet ndarje me minimum 3 raste per klase.")
else:
    train_df, val_df, test_df = best_result

    print("Best seed:", best_seed)
    print("Balance score:", round(best_score, 4))
    print("Training:", len(train_df))
    print("Validation:", len(val_df))
    print("Testing:", len(test_df))

Best seed: 436
Balance score: 0.032
Training: 803
Validation: 178
Testing: 172


In [11]:
if best_result is not None:
    distribution = pd.DataFrame({
        "Training": train_df["label"].value_counts(),
        "Validation": val_df["label"].value_counts(),
        "Testing": test_df["label"].value_counts()
    }).fillna(0).astype(int)

    display(distribution)

    print("Minimum samples per class:")
    print(distribution.min())

    train_groups = set(train_df["group_id"])
    val_groups = set(val_df["group_id"])
    test_groups = set(test_df["group_id"])

    print("\nGroup overlap:")
    print("Train-Val:", len(train_groups & val_groups))
    print("Train-Test:", len(train_groups & test_groups))
    print("Val-Test:", len(val_groups & test_groups))

,Training,Validation,Testing
label,,,
Acne,34,5,7
Arthritis,29,8,9
Bronchial Asthma,33,7,9
Cervical spondylosis,34,8,7
Chicken pox,33,8,8
Common Cold,31,10,8
Dengue,33,8,9
Dimorphic Hemorrhoids,26,7,8
Fungal infection,39,6,5


Minimum samples per class:
Training      26
Validation     4
Testing        4
dtype: int64

Group overlap:
Train-Val: 0
Train-Test: 0
Val-Test: 0


In [12]:
from pathlib import Path

processed_path = project_path / "data" / "processed"

processed_path.mkdir(parents=True, exist_ok=True)

train_df.to_csv(processed_path / "train.csv", index=False)
val_df.to_csv(processed_path / "validation.csv", index=False)
test_df.to_csv(processed_path / "test.csv", index=False)

print("Training saved:", len(train_df))
print("Validation saved:", len(val_df))
print("Testing saved:", len(test_df))

print("\nAll datasets saved successfully!")

Training saved: 803
Validation saved: 178
Testing saved: 172

All datasets saved successfully!
